In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-6-3-target-gene

Plot target-gene and proteomics validation results.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================

# ==========================================
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.5

# ==========================================

# ==========================================
file_path = input_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/2-CR/mmc2.xlsx")
sheet_name = "Figure 6E"

print("⏳ 1. 正在读取顶刊 Excel 原始数据 (这可能需要十几秒)...")

df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

print("🎯 2. 正在智能解析实验分组...")

row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 



col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]

col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]

col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]

print(f"✅ 列定位成功！\n   - 对照组(Control)在第 {col_ctrl} 列\n   - CR组(Low)在第 {col_cr} 列\n   - Rapa组在第 {col_rapa} 列")

# ==========================================

# ==========================================
print("\n🧬 3. 正在搜索纯金靶点并计算逆转幅度...")

genes = df_raw.iloc[3:, 0].astype(str).str.upper() 


target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6', 'LYZ2']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]

        val_ctrl = float(df_raw.iloc[row_idx, col_ctrl])
        val_cr = float(df_raw.iloc[row_idx, col_cr])
        val_rapa = float(df_raw.iloc[row_idx, col_rapa])
        

        if val_ctrl > 0:
            log2fc_cr = np.log2(val_cr / val_ctrl)
            log2fc_rapa = np.log2(val_rapa / val_ctrl)
            
            plot_data.append({'Protein': gene, 'Intervention': 'Caloric Restriction (CR)', 'Log2FC': log2fc_cr})
            plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
            found_genes.append(gene)
            print(f"   🌟 找到 {gene}: CR效应 = {log2fc_cr:.2f}, Rapa效应 = {log2fc_rapa:.2f}")

if not plot_data:
    print("❌ 警告：在这个表格里没有搜到目标基因。请检查靶点名是否在其他 Tab 页 (如 Figure 6C)。")
else:
    # ==========================================

    # ==========================================
    print(f"\n🎨 4. 开始绘制包含 {len(found_genes)} 个靶点的机制验证图...")
    df_plot = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(1.5 * len(found_genes), 4.5), dpi=150)
    

    sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
                palette={'Caloric Restriction (CR)': '#4C72B0', 'Rapamycin': '#55A868'},
                edgecolor='black', linewidth=1.2, ax=ax)
    

    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    

    ax.set_title("Cross-omics Validation:\nProteomic Reversibility by mTOR Targeting", fontweight='bold', fontsize=14, pad=15)
    ax.set_ylabel("Protein Expression\n(Log2FC vs Baseline Control)", fontweight='bold', fontsize=12)
    ax.set_xlabel("")
    

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(width=1.5, labelsize=12)
    
    plt.legend(title="", frameon=False, fontsize=10, loc='best')
    plt.tight_layout()
    

    plt.savefig("Figure_5G_Proteomics_Validation.pdf", bbox_inches='tight')
    plt.show()
    print("🎉 恭喜！完美的大图已生成并保存为 Figure_5G_Proteomics_Validation.pdf")

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================

# ==========================================
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.5

# ==========================================

# ==========================================

file_path = input_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/table.xlsx")

print("⏳ 1. 正在读取独立的母表数据...")

df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

print("🎯 2. 正在智能解析实验分组...")

row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 

try:


    col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]

    col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]

    col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]
    
    print(f"✅ 列定位成功！\n   - 对照组(Control)在第 {col_ctrl} 列\n   - CR组(Low)在第 {col_cr} 列\n   - Rapa组在第 {col_rapa} 列")
except IndexError:
    print("❌ 警告：未找到对应的分组列！请确认该表格向右滚动时，确实包含 Rapamycin 相关的列。")
    exit()

# ==========================================

# ==========================================
print("\n🧬 3. 正在搜索纯金靶点并计算逆转幅度...")

genes = df_raw.iloc[3:, 0].astype(str).str.upper() 

8.74

target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]

        try:
            val_ctrl = float(df_raw.iloc[row_idx, col_ctrl])
            val_cr = float(df_raw.iloc[row_idx, col_cr])
            val_rapa = float(df_raw.iloc[row_idx, col_rapa])
            

            if val_ctrl > 0:
                log2fc_cr = np.log2(val_cr / val_ctrl)
                log2fc_rapa = np.log2(val_rapa / val_ctrl)
                
                plot_data.append({'Protein': gene, 'Intervention': 'Caloric Restriction (CR)', 'Log2FC': log2fc_cr})
                plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
                found_genes.append(gene)
                print(f"   🌟 找到 {gene}: CR效应 = {log2fc_cr:.2f}, Rapa效应 = {log2fc_rapa:.2f}")
        except ValueError:
            print(f"   ⚠️ {gene} 的数据存在空值或非数字，跳过计算。")

if not plot_data:
    print("❌ 警告：在这个表格里依然没有搜到目标基因。")
else:
    # ==========================================

    # ==========================================
    print(f"\n🎨 4. 开始绘制包含 {len(found_genes)} 个靶点的机制验证图...")
    df_plot = pd.DataFrame(plot_data)
    

    fig, ax = plt.subplots(figsize=(max(4, 1.5 * len(found_genes)), 4.5), dpi=150)
    

    sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
                palette={'Caloric Restriction (CR)': '#4C72B0', 'Rapamycin': '#55A868'},
                edgecolor='black', linewidth=1.2, ax=ax)
    

    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    

    ax.set_title("Cross-omics Validation:\nProteomic Reversibility by mTOR Targeting", fontweight='bold', fontsize=14, pad=15)
    ax.set_ylabel("Protein Expression\n(Log2FC vs Baseline Control)", fontweight='bold', fontsize=12)
    ax.set_xlabel("")
    

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(width=1.5, labelsize=12)
    
    plt.legend(title="", frameon=False, fontsize=10, loc='best')
    plt.tight_layout()
    

    save_path = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/Proteomics_Validation.pdf")
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"🎉 恭喜！完美的大图已生成并保存至:\n👉 {save_path}")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================

# ==========================================
sns.reset_orig()


mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'


mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 7
mpl.rcParams['axes.labelsize'] = 7
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 5.5


mpl.rcParams['axes.linewidth'] = 0.5
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['ytick.major.width'] = 0.5
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

sc_dpi = 300

# ==========================================

# ==========================================
file_path = input_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR/table.xlsx")
output_dir = output_path("2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR")
os.makedirs(output_dir, exist_ok=True)

print("⏳ 正在读取母表数据并智能解析分组...")
df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

row_energy = df_raw.iloc[0].astype(str).str.strip().str.lower()
row_drug = df_raw.iloc[1].astype(str).str.strip().str.lower()
row_protein = df_raw.iloc[2].astype(str).str.strip() 

try:
    col_ctrl = df_raw.columns[(row_energy == 'standard') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    col_cr = df_raw.columns[(row_energy == 'low') & (row_drug == 'control') & (row_protein.str.startswith('19'))][0]
    col_rapa = df_raw.columns[(row_energy == 'standard') & (row_drug == 'rapamycin') & (row_protein.str.startswith('19'))][0]
except IndexError:
    sys.exit("❌ 未找到对应的分组列，请检查 Excel。")

# ==========================================

# ==========================================
genes = df_raw.iloc[3:, 0].astype(str).str.strip().str.upper() 
target_pool = ['HSPA8', 'CTSL', 'RPS10', 'RPS28', 'S100A6']
plot_data = []
found_genes = []

for gene in target_pool:
    idx = genes[genes == gene].index
    if len(idx) > 0:
        row_idx = idx[0]
        try:
            val_ctrl = float(df_raw.loc[row_idx, col_ctrl])
            val_cr = float(df_raw.loc[row_idx, col_cr])
            val_rapa = float(df_raw.loc[row_idx, col_rapa])
            
            pseudocount = 1e-6
            
            if val_ctrl > 0:
                log2fc_cr = np.log2((val_cr + pseudocount) / (val_ctrl + pseudocount))
                log2fc_rapa = np.log2((val_rapa + pseudocount) / (val_ctrl + pseudocount))
                
                plot_data.append({'Protein': gene, 'Intervention': 'CR', 'Log2FC': log2fc_cr})
                plot_data.append({'Protein': gene, 'Intervention': 'Rapamycin', 'Log2FC': log2fc_rapa})
                if gene not in found_genes: found_genes.append(gene)
        except ValueError:
            pass

if not plot_data:
    sys.exit("❌ 未搜到目标基因。")

# ==========================================

# ==========================================
df_plot = pd.DataFrame(plot_data)



fig, ax = plt.subplots(figsize=(3.0, 2.2), dpi=sc_dpi)


palette = {'CR': '#3C5488', 'Rapamycin': '#00A087'}


sns.barplot(data=df_plot, x='Protein', y='Log2FC', hue='Intervention', 
            palette=palette, edgecolor='black', linewidth=0.5, 
            width=0.7, ax=ax, saturation=1.0)


ax.axhline(0, color='#555555', linewidth=0.5, linestyle='--', zorder=0)


ax.set_ylabel(r"$\log_2$ FC (vs Control)", labelpad=4)
ax.set_xlabel("")


ax.tick_params(axis='both', length=2.5, pad=2)


ax.set_xticklabels(ax.get_xticklabels(), fontstyle='italic')


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.0, 1.15), 
          ncol=2, frameon=False, handletextpad=0.4, columnspacing=1.0)


# ax.set_title("Proteomic validation of mTOR targets", pad=15)

plt.tight_layout()


save_pdf = os.path.join(output_dir, "Figure_5G_MainJournal_Style.pdf")
save_svg = os.path.join(output_dir, "Figure_5G_MainJournal_Style.svg")

plt.savefig(save_pdf, bbox_inches='tight')
plt.savefig(save_svg, bbox_inches='tight', format='svg')
plt.close(fig)

print(f"✅ 主刊级正图已生成！请将此 PDF/SVG 直接拖入 AI 感受其精致度：\n👉 {save_pdf}")